# Phase 4: SVM Integration (Kaggle Version)
**CSC14120 - Parallel Programming**

## Overview

### Objectives
- Extract features using trained encoder (from Phase 3)
- Train SVM classifier on learned features
- Evaluate end-to-end classification performance

### Methodologies

#### Feature Extraction
- **Architecture**: Convolutional autoencoder encoder path
- **Input**: 32×32×3 CIFAR-10 images
- **Output**: 8,192-dimensional feature vectors (8×8×128 latent representation)
- **Method**: GPU-accelerated forward pass through encoder only

#### SVM Integration
- Using **ThunderSVM** (GPU-accelerated) or **cuML SVC** for fast training
- Features extracted via CUDA, then passed to SVM for training
- Binary feature files (`train_features.bin`, `test_features.bin`) for efficient I/O

#### Hyperparameter Selection
| Parameter | Value |
|:----------|:------|
| Kernel | RBF |
| C | 10-20 |
| gamma | 'scale' |
| Preprocessing | StandardScaler |
| Feature dim | 8192 |

### Target: 60-65% accuracy

In [ ]:
# Kiểm tra GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Copy project từ Kaggle dataset
import os
import glob
import shutil

input_dir = '/kaggle/input'
src_dirs = glob.glob(f'{input_dir}/**/src', recursive=True)

if src_dirs:
    project_dir = os.path.dirname(src_dirs[0])
    print(f"Found project at: {project_dir}")
    shutil.copytree(project_dir, '/kaggle/working/project', dirs_exist_ok=True)
    
    for root, dirs, _ in os.walk('/kaggle/working/project'):
        if 'src' in dirs:
            os.chdir(root)
            break
else:
    print("ERROR: Project not found")

print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Download CIFAR-10
import urllib.request
import tarfile

os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve(
        'https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 
        'data/cifar.tar.gz'
    )
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/

print('CIFAR-10 ready!')

## Load Pre-trained Weights

Sử dụng weights từ Phase 3 (BCE Loss) đã train.

In [ ]:
weights_file = "phase3_bce.weights"

if not os.path.exists(weights_file):
    weight_files = glob.glob('/kaggle/input/**/*.weights', recursive=True)
    if weight_files:
        shutil.copy(weight_files[0], weights_file)
        print(f"Copied weights from: {weight_files[0]}")
    else:
        print("ERROR: No weights file found!")
else:
    print(f"Found weights: {weights_file}")

!ls -lh {weights_file}

## Build Feature Extractor

**Key Code Snippet - Feature Extraction:**

The feature extractor uses the trained encoder to extract 8192-dim features:

```cpp
// From main_phase4.cu - Encoder forward pass
void extract_features(GPUAutoencoder& ae, const float* images, 
                      float* features, int n_samples) {
    for (int i = 0; i < n_samples; i += batch_size) {
        // Forward pass through encoder only
        ae.encoder_forward(batch_images);
        // Copy latent representation (8x8x128 = 8192 dims)
        cudaMemcpy(features + i * 8192, ae.get_latent(), 
                   8192 * sizeof(float), cudaMemcpyDeviceToHost);
    }
}
```

The extracted features are saved to binary files for efficient loading in Python.

In [ ]:
# Build feature extractor
print("Building LIBSVM...")
!git clone --depth 1 https://github.com/cjlin1/libsvm.git libsvm_src 2>/dev/null || echo "Already cloned"
!cd libsvm_src && make lib
!cp libsvm_src/svm.h include/ 2>/dev/null || true
!cp libsvm_src/svm.cpp src/ 2>/dev/null || true

print("\nBuilding feature_extractor...")
!nvcc -O3 -std=c++17 -arch=sm_70 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DWITH_SVM -DWITH_LIBSVM \
    -Iinclude -Ilibsvm_src -lcublas -lcudnn \
    -o feature_extractor \
    src/main_phase4.cu src/layers_gpu.cu src/gpu_autoencoder.cu \
    src/layers_gpu_opt.cu src/dataset.cpp src/svm_wrapper.cpp libsvm_src/svm.cpp

!if [ -f feature_extractor ]; then echo "Build SUCCESS!"; else echo "Build FAILED!"; fi

In [ ]:
# Extract features
import time

print("Extracting features with GPU autoencoder...")
start = time.time()
!./feature_extractor --data data --weights phase3_bce.weights --extract-only
print(f"\nFeature extraction: {time.time() - start:.2f}s")
!ls -lh *.bin 2>/dev/null || echo "No .bin files"

In [ ]:
# Load features and labels
import numpy as np
import struct

feature_dim = 8192

# Load features
print("Loading features...")
train_features = np.fromfile('train_features.bin', dtype=np.float32).reshape(-1, feature_dim)
test_features = np.fromfile('test_features.bin', dtype=np.float32).reshape(-1, feature_dim)

print(f"Train features: {train_features.shape}")
print(f"Test features: {test_features.shape}")

# Load labels
def load_cifar10_labels(data_dir):
    train_labels, test_labels = [], []
    for i in range(1, 6):
        with open(f"{data_dir}/data_batch_{i}.bin", 'rb') as f:
            for _ in range(10000):
                train_labels.append(struct.unpack('B', f.read(1))[0])
                f.read(3072)
    with open(f"{data_dir}/test_batch.bin", 'rb') as f:
        for _ in range(10000):
            test_labels.append(struct.unpack('B', f.read(1))[0])
            f.read(3072)
    return np.array(train_labels), np.array(test_labels)

train_labels, test_labels = load_cifar10_labels('data')
print(f"Labels loaded: {len(train_labels)} train, {len(test_labels)} test")

In [ ]:
# Install RAPIDS (cuML) for GPU-accelerated preprocessing
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/env-check.py

## Feature Preprocessing

Standardize features (zero mean, unit variance) for better SVM performance.

In [ ]:
import gc
import cupy as cp

# Use cuML StandardScaler
try:
    from cuml.preprocessing import StandardScaler
    USE_CUML = True
    print("Using cuML (GPU-accelerated)")
except ImportError:
    from sklearn.preprocessing import StandardScaler
    USE_CUML = False
    print("cuML not available, using sklearn")

print("="*60)
print("FEATURE PREPROCESSING")
print("="*60)

# Standardize (zero mean, unit variance)
print("\n1. Standardizing features...")
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features)

if USE_CUML:
    print(f"   Train mean: {float(cp.mean(train_features_scaled)):.6f}, std: {float(cp.std(train_features_scaled)):.6f}")
else:
    print(f"   Train mean: {train_features_scaled.mean():.6f}, std: {train_features_scaled.std():.6f}")

print(f"   Feature dim: {train_features_scaled.shape[1]}")

# Use scaled features
train_features_final = train_features_scaled
test_features_final = test_features_scaled

# Convert to numpy if needed for compatibility
if USE_CUML:
    train_features_final = cp.asnumpy(train_features_final) if hasattr(train_features_final, 'get') else train_features_final
    test_features_final = cp.asnumpy(test_features_final) if hasattr(test_features_final, 'get') else test_features_final

# Free memory
del train_features, test_features
gc.collect()
if USE_CUML:
    cp.get_default_memory_pool().free_all_blocks()
print("\nMemory freed!")

## Stratified Sampling

Use stratified sampling to ensure balanced class distribution.

In [ ]:
# Use all 50000 samples for best accuracy
TRAIN_SAMPLES = 50000
samples_per_class = TRAIN_SAMPLES // 10

print(f"Stratified sampling: {samples_per_class} per class = {TRAIN_SAMPLES} total")

train_indices = []
for c in range(10):
    class_indices = np.where(train_labels == c)[0]
    np.random.seed(42)  # Reproducibility
    selected = np.random.choice(class_indices, size=samples_per_class, replace=False)
    train_indices.extend(selected)

train_indices = np.array(train_indices)
np.random.shuffle(train_indices)  # Shuffle for good measure

X_train = train_features_final[train_indices]
y_train = train_labels[train_indices]

X_test = test_features_final
y_test = test_labels

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class distribution: {np.bincount(y_train)}")

## Build ThunderSVM (GPU-accelerated SVM)

ThunderSVM provides significant speedup over CPU-based LIBSVM.

In [ ]:
import sys

print("Building ThunderSVM from source...")

# Cleanup previous attempts
!rm -rf thundersvm

# Clone
!git clone --depth 1 https://github.com/Xtra-Computing/thundersvm.git

# Build with explicit CUDA support
!cd thundersvm && mkdir -p build && cd build && cmake .. -DUSE_CUDA=ON -DCMAKE_BUILD_TYPE=Release && make -j4

# Install Python bindings
!cd thundersvm/python && pip install -e .

# Verify import
USE_THUNDER = False
try:
    sys.path.insert(0, 'thundersvm/python')
    from thundersvm import SVC as ThunderSVC
    print("\n" + "="*50)
    print("ThunderSVM installed successfully!")
    print("="*50)
    USE_THUNDER = True
except ImportError as e:
    print(f"\nFailed to import ThunderSVM: {e}")
    print("Will fall back to sklearn SVC (LibSVM)")
    USE_THUNDER = False

## Train SVM Classifier

In [ ]:
import time
from sklearn.metrics import accuracy_score, classification_report

# SVM hyperparameters
C_VALUE = 10
GAMMA = 'scale'

print("="*60)
print("SVM TRAINING")
print("="*60)
print(f"C = {C_VALUE}, gamma = {GAMMA}")
print(f"Training samples: {len(X_train)}")

if USE_THUNDER:
    print("\nUsing ThunderSVM (GPU)...")
    svm = ThunderSVC(kernel='rbf', C=C_VALUE, gamma=GAMMA)
else:
    print("\nUsing sklearn SVC (CPU)...")
    from sklearn.svm import SVC
    svm = SVC(kernel='rbf', C=C_VALUE, gamma=GAMMA, cache_size=2000)

start = time.time()
svm.fit(X_train, y_train)
train_time = time.time() - start
print(f"Training time: {train_time:.2f}s")

# Predict
print("\nPredicting...")
start = time.time()
y_pred = svm.predict(X_test)
pred_time = time.time() - start
print(f"Prediction time: {pred_time:.2f}s")

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n" + "="*60)
print(f"TEST ACCURACY: {accuracy*100:.2f}%")
print("="*60)

In [ ]:
# Detailed classification report
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CIFAR10_CLASSES))

In [ ]:
# Confusion matrix visualization
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix (Accuracy: {accuracy*100:.2f}%)')
plt.tight_layout()
plt.savefig('phase4_confusion_matrix.png', dpi=150)
plt.show()

## Summary

### Phase 4 Results
- **Feature Extraction**: GPU-accelerated encoder forward pass
- **Feature Dimension**: 8192 (8×8×128 latent space)
- **Preprocessing**: StandardScaler (zero mean, unit variance)
- **SVM**: RBF kernel with C=10, gamma='scale'
- **Target Accuracy**: 60-65%

In [ ]:
# Final summary
print("="*60)
print("PHASE 4 SUMMARY")
print("="*60)
print(f"Feature dimension: {X_train.shape[1]}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"SVM kernel: RBF (C={C_VALUE}, gamma={GAMMA})")
print(f"Training time: {train_time:.2f}s")
print(f"Prediction time: {pred_time:.2f}s")
print(f"Test accuracy: {accuracy*100:.2f}%")
print("="*60)